# Course 5 — Feature Selection I

Best-subset selection, forward/backward stepwise, and the four
model-size criteria: AIC, BIC, Mallow's Cp, adjusted R².

**Sections**
1. The combinatorial cost of best-subset (0:00–0:30)
2. Forward stepwise from scratch (0:30–1:00)
3. AIC, BIC, Cp, adjusted R² (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error

hitters = load_dataset('hitters').dropna(subset=['Salary']).reset_index(drop=True)
hitters = pd.get_dummies(hitters, columns=['League', 'Division', 'NewLeague'],
                          drop_first=True, dtype=float)
y = hitters['Salary'].to_numpy()
X = hitters.drop(columns=['Salary'])
print(X.shape, X.columns.tolist()[:6], '...')


## 1. Best-subset is exponential

With p = 19 predictors, $2^{19} = 524{,}288$ subsets. We will not enumerate
all of them; we will enumerate a few to see the shape of the problem.

In [ ]:
# Best-subset on the first 5 predictors only -> 2^5 - 1 = 31 subsets
small = X.columns[:5].tolist()
best_by_size = {}
for k in range(1, len(small) + 1):
    best_rss = float('inf'); best_combo = None
    for combo in combinations(small, k):
        Xs = X[list(combo)].to_numpy()
        m = LinearRegression().fit(Xs, y)
        rss = ((y - m.predict(Xs)) ** 2).sum()
        if rss < best_rss:
            best_rss, best_combo = rss, combo
    best_by_size[k] = (best_combo, best_rss)
    print(f'k={k}: RSS={best_rss:.0f}  cols={best_combo}')


RSS always drops as we add predictors — that's why we *can't* pick a
model size by raw RSS. We need a criterion that penalizes size.

## 2. Forward stepwise from scratch

In [ ]:
def forward_stepwise(X: pd.DataFrame, y: np.ndarray, n_folds: int = 5) -> list[tuple[str, float]]:
    remaining = list(X.columns); chosen: list[str] = []; trace = []
    cv = KFold(n_folds, shuffle=True, random_state=0)
    while remaining:
        best_col, best_mse = None, float('inf')
        for col in remaining:
            cols = chosen + [col]
            scores = cross_val_score(LinearRegression(), X[cols].to_numpy(), y,
                                     cv=cv, scoring='neg_mean_squared_error')
            mse = -scores.mean()
            if mse < best_mse:
                best_mse, best_col = mse, col
        chosen.append(best_col); remaining.remove(best_col)
        trace.append((best_col, best_mse))
    return trace

trace = forward_stepwise(X, y)
for i, (col, mse) in enumerate(trace, 1):
    print(f'{i:2d}. add {col:15s}  CV-MSE = {mse:.0f}')


In [ ]:
sizes = list(range(1, len(trace) + 1))
mses = [m for _, m in trace]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, mses, marker='o')
best = int(np.argmin(mses)) + 1
ax.axvline(best, color='C3', linestyle='--', label=f'best = {best}')
ax.set_xlabel('model size'); ax.set_ylabel('5-fold CV MSE')
ax.set_title('Forward stepwise on Hitters'); ax.legend(); plt.show()


## 3. AIC, BIC, Cp, adjusted R²

In [ ]:
def info_criteria(X: np.ndarray, y: np.ndarray) -> dict:
    n, p = X.shape
    m = LinearRegression().fit(X, y)
    rss = ((y - m.predict(X)) ** 2).sum()
    r2 = m.score(X, y)
    sigma2 = rss / (n - p - 1)
    return dict(
        aic = n * np.log(rss / n) + 2 * p,
        bic = n * np.log(rss / n) + p * np.log(n),
        cp  = (rss + 2 * p * sigma2) / n,
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1),
    )

rows = []
chosen = []
for col, _ in trace:
    chosen.append(col)
    ic = info_criteria(X[chosen].to_numpy(), y)
    ic['size'] = len(chosen)
    rows.append(ic)
ic_df = pd.DataFrame(rows).set_index('size')
print(ic_df.round(2).to_string())


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, col in zip(axes, ['aic', 'bic', 'cp', 'adj_r2']):
    ax.plot(ic_df.index, ic_df[col], marker='o')
    pick = ic_df[col].idxmin() if col != 'adj_r2' else ic_df[col].idxmax()
    ax.axvline(pick, color='C3', linestyle='--')
    ax.set_title(f'{col} (picks size {pick})')
    ax.set_xlabel('model size')
plt.tight_layout(); plt.show()


### Recap
- Best-subset is exponential; stepwise is greedy but cheap and usually fine.
- AIC / BIC / Cp / adj-R² each penalize size differently. BIC is the most
  conservative; adjusted R² is the loosest.
- Pick a single criterion *before* you look at the answers.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Load `titanic`, drop rows with missing `fare`, one-hot
`sex`, `embarked`, `class`. Implement forward stepwise to predict
`fare` and plot the CV-MSE curve.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
df = load_dataset('titanic').dropna(subset=['fare']).reset_index(drop=True)
df = pd.get_dummies(df[['fare','pclass','sex','age','sibsp','parch','embarked','class']].fillna(df.median(numeric_only=True)),
                     columns=['sex','embarked','class'], drop_first=True, dtype=float)
y = df['fare'].to_numpy(); X = df.drop(columns=['fare'])
cv = KFold(5, shuffle=True, random_state=0)
remaining = list(X.columns); chosen = []; mses = []
while remaining:
    best_col, best_mse = None, float('inf')
    for col in remaining:
        s = -cross_val_score(LinearRegression(), X[chosen+[col]].to_numpy(), y, cv=cv,
                              scoring='neg_mean_squared_error').mean()
        if s < best_mse: best_mse, best_col = s, col
    chosen.append(best_col); remaining.remove(best_col); mses.append(best_mse)
fig, ax = plt.subplots(figsize=(6,3.5))
ax.plot(range(1, len(mses)+1), mses, marker='o')
ax.set_xlabel('size'); ax.set_ylabel('CV-MSE'); plt.show()
print('order added:', chosen)


---

## Exercise 2

**Task 2.** Which model size does *BIC* pick on the titanic-fare problem?

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
n = len(y)
rows = []
for k in range(1, len(chosen)+1):
    Xk = X[chosen[:k]].to_numpy()
    m = LinearRegression().fit(Xk, y)
    rss = ((y - m.predict(Xk))**2).sum()
    rows.append((k, n*np.log(rss/n) + k*np.log(n)))
import pandas as pd
bic_df = pd.DataFrame(rows, columns=['size', 'bic']).set_index('size')
print('BIC picks size =', bic_df['bic'].idxmin())


---

## Exercise 3

**Task 3.** Does CV pick the same size as BIC? Print both. Which one
do you trust more, and why?

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
cv_best = mses.index(min(mses)) + 1
bic_best = bic_df['bic'].idxmin()
print(f'CV picks size = {cv_best}, BIC picks size = {bic_best}')
print('CV is the more honest test-error estimate;')
print('BIC is faster but assumes the linear model is correctly specified.')
